In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data")

game_file = next(DATA_DIR.glob("game_data_public.*.csv.gz"))
replay_file = next(DATA_DIR.glob("replay_data_public.*.csv.gz"))

print("Game file:", game_file.name)
print("Replay file:", replay_file.name)

# Load a manageable sample first
GAME_KEY = ["draft_id", "match_number", "game_number"]

game_df = pd.read_csv(
    game_file,
    nrows=100,
)

wanted_keys = set(
    map(
        tuple,
        game_df[GAME_KEY].itertuples(index=False, name=None)
    )
)

replay_parts = []

for chunk in pd.read_csv(
    replay_file,
    chunksize=10_000,
    low_memory=False,
):
    keys = list(
        zip(
            chunk["draft_id"],
            chunk["match_number"],
            chunk["game_number"],
        )
    )

    mask = [key in wanted_keys for key in keys]

    if any(mask):
        replay_parts.append(chunk.loc[mask])

replay_df = pd.concat(
    replay_parts,
    ignore_index=True,
)

print("game rows:", len(game_df))
print("replay rows:", len(replay_df))

Game file: game_data_public.Cube_-_Powered.PremierDraft.csv.gz
Replay file: replay_data_public.Cube_-_Powered.PremierDraft.csv.gz
game rows: 100
replay rows: 100


In [26]:
import duckdb
import numpy as np
import pandas as pd


DB_PATH = "../data/17lands.duckdb"

con = duckdb.connect(DB_PATH, read_only=True)

df = con.execute("""
    SELECT
        c.card_id,
        c.card_name,
        e.embedding
    FROM deck_ppmi_svd_embeddings e
    JOIN cards c
        ON c.card_id = e.card_id
    WHERE e.has_embedding
""").df()

con.close()


# Convert DuckDB arrays into a dense NumPy matrix.
vectors = np.vstack(df["embedding"].to_numpy())

name_to_idx = {
    name: i
    for i, name in enumerate(df["card_name"])
}


def similar_cards(card_name, n=15):
    idx = name_to_idx[card_name]

    # Embeddings were already L2-normalized when generated,
    # so the dot product is cosine similarity.
    similarities = vectors @ vectors[idx]

    order = np.argsort(-similarities)

    rows = []

    for other_idx in order:
        if other_idx == idx:
            continue

        rows.append({
            "card": df.iloc[other_idx]["card_name"],
            "similarity": float(similarities[other_idx]),
        })

        if len(rows) == n:
            break

    return pd.DataFrame(rows)


similar_cards("Lightning Bolt", 15)

,card,similarity
0,"Magda, Brazen Outlaw",0.993367
1,Broadside Bombardiers,0.992143
2,Burst Lightning,0.987058
3,Chain Lightning,0.987042
4,Generous Plunderer,0.985830
5,"Ragavan, Nimble Pilferer",0.985460
6,"Inti, Seneschal of the Sun",0.983822
7,Bonecrusher Giant,0.983803
8,Flame Slash,0.983150
9,Fury,0.983101


In [29]:
similar_cards("Minsc & Boo, Timeless Heroes", 10)

,card,similarity
0,Copperline Gorge,0.973455
1,Thornspire Verge,0.965556
2,Stomping Ground,0.959384
3,Commercial District,0.958881
4,Taiga,0.953067
5,Bloodbraid Elf,0.942268
6,Wrenn and Six,0.933501
7,Questing Druid,0.931828
8,Wooded Foothills,0.928261
9,Jetmir's Garden,0.925385
